In [24]:
import json
import networkx as nx
import numpy as np
import os
from src.authorship_util import get_authors_api
from pathlib import Path 

INSTITUICOES = ['uft', 'ufnt', 'ceulp', 'ifto', 'unitins', 'tocantins']

In [14]:
def load_network_data(institution, year):
    """Carrega os dados da rede a partir do arquivo JSON"""
    file_path = f'../results/metrics/{institution}/{institution}_{year}.json'
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

In [15]:
ids = set()
for instituicao in INSTITUICOES:
    data = load_network_data(instituicao, 2024)
    degrees = data['degrees']
    betweenness_centrality = data['betweenness_centrality']
    closeness_centrality = data['closeness_centrality']
    eigenvector_centrality = data['eigenvector_centrality']
    metrics = {
        'degrees' : sorted(degrees.items(), key=lambda item: item[1], reverse=True),
        'betweenness_centrality' : sorted(betweenness_centrality.items(), key=lambda item: item[1], reverse=True),
        'closeness_centrality' : sorted(closeness_centrality.items(), key=lambda item: item[1], reverse=True),
        'eigenvector_centrality' : sorted(eigenvector_centrality.items(), key=lambda item: item[1], reverse=True),
    }
    for _, metric in metrics.items():
        ids.update([item[0] for item in metric[:10]])

In [16]:
authors = []

In [17]:
def chunk_list(input_list, chunk_size):
    chunks = []
    for i in range(0, len(input_list), chunk_size):
        chunks.append(input_list[i:i + chunk_size])
    return chunks

chunk_size = 10
chunks = chunk_list(list(ids), chunk_size)
for chunk in chunks:
    authors.extend(get_authors_api(chunk))

In [25]:
with open(Path("../results/top_authors_info.json"), "w") as f:
    json.dump(authors, f)

In [2]:
import json
from collections import defaultdict

# Load the network metrics data for each institution
def load_network_data(institution, year=2024):
    """Carrega os dados da rede a partir do arquivo JSON"""
    # NOTA: Pode ser necessário ajustar o caminho do arquivo para corresponder à sua estrutura de diretórios.
    # Exemplo: file_path = f'./results/metrics/{institution}/{institution}_{year}.json'
    file_path = f'../results/metrics/{institution}/{institution}_{year}.json'
    with open(file_path, 'r') as f:
        return json.load(f)

# Load all institutions data
INSTITUICOES = ['uft', 'ufnt', 'ceulp', 'ifto', 'unitins', 'tocantins']
institution_metrics = {}
for instituicao in INSTITUICOES:
    try:
        institution_metrics[instituicao] = load_network_data(instituicao)
    except FileNotFoundError:
        print(f"Aviso: Arquivo de dados não encontrado para '{instituicao}'. Pulando esta instituição.")
        continue

# Load authors data
# NOTA: Pode ser necessário ajustar o caminho do arquivo para corresponder à sua estrutura de diretórios.
try:
    with open('../results/top_authors_info.json', 'r') as f:
        authors_data = json.load(f)
except FileNotFoundError:
    print("Aviso: Arquivo 'top_authors_info.json' não encontrado. Os dados dos autores não estarão completos.")
    authors_data = []


# Create a dictionary for quick author lookup
authors_dict = {author['id']: author for author in authors_data}

# Map institution codes to full names
INSTITUTION_NAMES = {
    'uft': 'Universidade Federal do Tocantins',
    'ufnt': 'Universidade Federal do Norte do Tocantins',
    'ceulp': 'Centro Universitário Luterano de Palmas',
    'ifto': 'Instituto Federal do Tocantins',
    'unitins': 'Universidade do Tocantins',
    'tocantins': 'Estado do Tocantins' # Ajustado para clareza
}

# Define the metrics we'll use (translated to Portuguese)
metrics = {
    'degrees': 'Grau',
    'betweenness_centrality': 'Intermediação',
    'closeness_centrality': 'Proximidade'
}

# Function to format numbers with comma as decimal separator
def format_number(value):
    if isinstance(value, float):
        # Formata com 4 casas decimais e substitui ponto por vírgula
        return f"{value:.4f}".replace(".", ",")
    elif isinstance(value, int):
        return f"{value}"
    return str(value)

# Generate LaTeX tables for each institution
for inst_code, metrics_data in institution_metrics.items():
    institution_name = INSTITUTION_NAMES.get(inst_code, inst_code.upper())
    
    print(f"% Tabela para {institution_name}")
    print("\\begin{table}[!htpb]")
    print("    \\centering")
    caption_name = institution_name.replace('Universidade', 'Univ.').replace('Federal', 'Fed.').replace('Centro Universitário', 'Centro Univ.')
    print(f"    \\caption{{Top 3 pesquisadores da {caption_name} por métricas de centralidade.}}")
    print(f"    \\label{{tb:top_centrality_{inst_code}}}")
    print("    \\footnotesize")
    print("    \\scalebox{0.8}{")
    print("        \\begin{tabular}{ >{\\centering\\arraybackslash}m{2.2cm} | L{6cm} | C{2cm} | C{1.2cm} | C{1.2cm} | C{2cm} }")
    print("        \\hline")
    print("        \\textbf{Métrica} & \\textbf{Pesquisador} & \\textbf{Trabalhos} & \\textbf{h-index} & \\textbf{Valor} & \\textbf{Curso} \\\\ \\hline")
    
    metric_keys = list(metrics.keys())
    # Para cada métrica, obtém os 3 melhores autores
    for metric_idx, metric in enumerate(metric_keys):
        metric_name = metrics[metric]
        
        metric_values = metrics_data.get(metric, {})
        sorted_items = sorted(metric_values.items(), key=lambda x: x[1], reverse=True)[:3]
        
        # Define se o bloco de métrica é par ou ímpar (começando do 0)
        is_even_block = (metric_idx % 2 == 0)
        
        # Itera sobre os 3 melhores autores para a métrica (índice da linha local i = 0, 1, 2)
        for i in range(3):
            if i < len(sorted_items):
                author_id, value = sorted_items[i]
                author = authors_dict.get(author_id, {'display_name': 'Desconhecido', 'works_count': '?', 'summary_stats': {'h_index': '?'}})
                h_index = author.get('summary_stats', {}).get('h_index', '?')
                formatted_value = format_number(value)
                
                # Lógica para determinar se a linha atual deve ser colorida
                color_row = False
                if is_even_block and (i == 0 or i == 2):  # Bloco par (1º, 3º, ...): colore a 1ª e 3ª linha
                    color_row = True
                elif not is_even_block and i == 1:      # Bloco ímpar (2º, 4º, ...): colore a 2ª linha
                    color_row = True
                
                # Prepara as células com ou sem \cellcolor
                if color_row:
                    cells = (f"\\cellcolor{{lightgray}}{author['display_name']}",
                             f"\\cellcolor{{lightgray}}{author.get('works_count', '?')}",
                             f"\\cellcolor{{lightgray}}{h_index}",
                             f"\\cellcolor{{lightgray}}{formatted_value}",
                             f"\\cellcolor{{lightgray}}")
                else:
                    cells = (f"{author['display_name']}",
                             f"{author.get('works_count', '?')}",
                             f"{h_index}",
                             f"{formatted_value}",
                             "")
                
                # Imprime a linha completa
                if i == 0:
                    # A primeira linha usa multirow
                    print(f"        \\multirow{{3}}{{*}}{{{metric_name}}} & {cells[0]} & {cells[1]} & {cells[2]} & {cells[3]} & {cells[4]} \\\\")
                else:
                    # As linhas seguintes
                    print(f"         & {cells[0]} & {cells[1]} & {cells[2]} & {cells[3]} & {cells[4]} \\\\")
            else:
                # Caso não haja 3 autores, preenche com linhas vazias
                if i == 0:
                     print(f"        \\multirow{{3}}{{*}}{{{metric_name}}} & & & & & \\\\")
                else:
                    print("         & & & & & \\\\")
        
        # Adiciona \hline após cada bloco de métrica, exceto o último
        if metric_idx < len(metric_keys) - 1:
            print("        \\hline")

    print("        \\hline")
    print("        \\end{tabular}")
    print("    }")
    print("\\end{table}")
    print("\n\n")

% Tabela para Universidade Federal do Tocantins
\begin{table}[!htpb]
    \centering
    \caption{Top 3 pesquisadores da Univ. Fed. do Tocantins por métricas de centralidade.}
    \label{tb:top_centrality_uft}
    \footnotesize
    \scalebox{0.8}{
        \begin{tabular}{ >{\centering\arraybackslash}m{2.2cm} | L{6cm} | C{2cm} | C{1.2cm} | C{1.2cm} | C{2cm} }
        \hline
        \textbf{Métrica} & \textbf{Pesquisador} & \textbf{Trabalhos} & \textbf{h-index} & \textbf{Valor} & \textbf{Curso} \\ \hline
        \multirow{3}{*}{Grau} & \cellcolor{lightgray}Gil Rodrigues dos Santos & \cellcolor{lightgray}333 & \cellcolor{lightgray}20 & \cellcolor{lightgray}491 & \cellcolor{lightgray} \\
         & Renato Almeida Sarmento & 235 & 28 & 399 &  \\
         & \cellcolor{lightgray}Raimundo Wagner de Souza Aguiar & \cellcolor{lightgray}159 & \cellcolor{lightgray}21 & \cellcolor{lightgray}380 & \cellcolor{lightgray} \\
        \hline
        \multirow{3}{*}{Intermediação} & Raphael Sânzio Pimenta 